In [ ]:
!pip install -q ultralytics albumentations opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.8 MB/s eta 0:00:00


In [ ]:
import zipfile
import os

os.makedirs('/content/dataset1', exist_ok=True)
os.makedirs('/content/dataset2', exist_ok=True)

# Unzip Dataset 1
with zipfile.ZipFile('/content/YOLOv8_dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset1')
print("✅ Extracted YOLOv8_dataset.zip to /content/dataset1")

# Unzip Dataset 2
with zipfile.ZipFile('/content/dataset_like_dislike_rock.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset2')
print("✅ Extracted dataset_like_dislike_rock.zip to /content/dataset2")

✅ Extracted YOLOv8_dataset.zip to /content/dataset1
✅ Extracted dataset_like_dislike_rock.zip to /content/dataset2


In [ ]:
import os
import yaml

# Define total 5 combined classes
class_names = {
    0: 'okay',
    1: 'spiderman',
    2: 'like',
    3: 'dislike',
    4: 'rock'
}

# Shift Dataset 2 class indices by +2 (0->2, 1->3, 2->4)
def remap_dataset2_labels(labels_dir, shift=2):
    if not os.path.exists(labels_dir):
        return
    for root, _, files in os.walk(labels_dir):
        for file in files:
            if file.endswith('.txt') and file != 'classes.txt':
                file_path = os.path.join(root, file)
                new_lines = []
                with open(file_path, 'r') as f:
                    lines = f.readlines()
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        original_class = int(parts[0])
                        # Only shift if original class index is 0, 1, or 2
                        if original_class < 3:
                            parts[0] = str(original_class + shift)
                        new_lines.append(" ".join(parts) + "\n")
                with open(file_path, 'w') as f:
                    f.writelines(new_lines)

# Remap labels inside dataset2
remap_dataset2_labels('/content/dataset2')
print("✅ Remapped Dataset 2 label IDs successfully (+2 offset).")

# Auto-locate image directories
def find_image_dirs(base_path):
    train_paths, val_paths = [], []
    for root, dirs, _ in os.walk(base_path):
        if 'images' in dirs:
            img_dir = os.path.join(root, 'images')
            if os.path.exists(os.path.join(img_dir, 'train')):
                train_paths.append(os.path.join(img_dir, 'train'))
            if os.path.exists(os.path.join(img_dir, 'val')):
                val_paths.append(os.path.join(img_dir, 'val'))
            if not train_paths:
                train_paths.append(img_dir)
                val_paths.append(img_dir)
    return train_paths, val_paths

train_d1, val_d1 = find_image_dirs('/content/dataset1')
train_d2, val_d2 = find_image_dirs('/content/dataset2')

# Create YOLO multi-dataset config
data_yaml = {
    'path': '/content',
    'train': train_d1 + train_d2,
    'val': (val_d1 + val_d2) if (val_d1 and val_d2) else (train_d1 + train_d2),
    'names': class_names
}

yaml_path = '/content/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"\n✅ Generated {yaml_path} successfully!")
print("Configured Classes:", class_names)

✅ Remapped Dataset 2 label IDs successfully (+2 offset).

✅ Generated /content/data.yaml successfully!
Configured Classes: {0: 'okay', 1: 'spiderman', 2: 'like', 3: 'dislike', 4: 'rock'}


In [ ]:
import os
import yaml
import zipfile

# 1. Unzip Datasets
os.makedirs('/content/dataset1', exist_ok=True)
os.makedirs('/content/dataset2', exist_ok=True)

with zipfile.ZipFile('/content/YOLOv8_dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset1')

with zipfile.ZipFile('/content/dataset_like_dislike_rock.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset2')

# 2. Shift Dataset 2 Class IDs by +2 (0->2, 1->3, 2->4)
def remap_dataset2_labels(base_dir, shift=2):
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.txt') and file != 'classes.txt':
                file_path = os.path.join(root, file)
                new_lines = []
                with open(file_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            orig_cls = int(parts[0])
                            if orig_cls < 3:
                                parts[0] = str(orig_cls + shift)
                            new_lines.append(" ".join(parts) + "\n")
                with open(file_path, 'w') as f:
                    f.writelines(new_lines)

remap_dataset2_labels('/content/dataset2')

# 3. Find ALL image directories automatically across both datasets
def get_all_image_directories(base_path):
    image_dirs = set()
    for root, _, files in os.walk(base_path):
        if any(f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')) for f in files):
            image_dirs.add(root)
    return list(image_dirs)

d1_images = get_all_image_directories('/content/dataset1')
d2_images = get_all_image_directories('/content/dataset2')

train_image_paths = d1_images + d2_images

# Ensure images were found
if not train_image_paths:
    raise ValueError("❌ Still no images found! Check if the zip files were uploaded properly to /content.")

# 4. Generate data.yaml
data_yaml = {
    'path': '/content',
    'train': train_image_paths,
    'val': train_image_paths,  # Fallback to train paths for validation if separate val split isn't present
    'names': {
        0: 'okay',
        1: 'spiderman',
        2: 'like',
        3: 'dislike',
        4: 'rock'
    }
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("✅ Successfully generated /content/data.yaml")
print(f"Found {len(train_image_paths)} image directories across both datasets.")

✅ Successfully generated /content/data.yaml
Found 3 image directories across both datasets.


In [ ]:
from ultralytics import YOLO

# Load pretrained model
model = YOLO('yolov8n.pt')

# Train model
results = model.train(
    data='/content/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='gesture_5classes_augmented',

    # --- AUGMENTATION PARAMETERS ---
    degrees=15.0,     # Rotation (+/- 15 degrees)
    fliplr=0.5,       # Horizontal Flip
    flipud=0.2,       # Vertical Flip
    hsv_h=0.015,      # Hue Jitter
    hsv_s=0.7,        # Saturation Jitter
    hsv_v=0.4,        # Brightness Jitter
    erasing=0.1,      # Cutout / Random Erasing
    perspective=0.001 # Perspective tilt
)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=15.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.1, exist_ok=False, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=gesture_5classes_augmented, nbs=64, nms=Fal

In [ ]:
from google.colab import files

# Path to your trained weights file
# (Replace 'gesture_5classes_augmented' with your run name if different)
weight_path = '/content/runs/gesture_5classes_augmented/weights/best.pt'

# Trigger browser download
files.download(weight_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>